In [2]:
import joblib
import pandas as pd

In [3]:
df = pd.read_csv("final.csv")
df = df.drop("Unnamed: 0", axis=1)
df.head()

,age_days,daily_usage_hours,avg_temperature_c,avg_humidity_percent,vibration_level,signal_strength_dbm,avg_power_consumption_w,firmware_version,camera_type,location_type,...,old_camera,high_temp,high_humidity,weak_signal,high_usage,power_per_hour,temp_humidity,age_vibration,signal_quality,risk_score
0,119.0,16.500000,33.2,68.8,0.39,-55.3,12.0,7,PTZ,Outdoor_Commercial,...,0,0,0,0,0,0.727273,2284.16,0.127151,44.7,0
1,678.0,17.083333,16.9,73.6,0.55,-57.8,14.0,8,Bullet,Warehouse,...,0,0,0,0,0,0.819512,1243.84,1.021644,42.2,0
2,552.0,17.600000,22.6,72.2,0.05,-52.0,13.6,3,Thermal,Indoor,...,0,0,0,0,0,0.772727,1631.72,0.075616,48.0,0
3,1101.5,17.216667,25.9,77.5,0.45,-52.8,16.7,10,Bullet,Indoor,...,1,0,0,0,0,0.969990,2007.25,1.358014,47.2,1
4,339.0,17.350000,23.8,92.2,0.29,-52.8,14.9,7,Bullet,Indoor,...,0,0,1,0,0,0.858790,2194.36,0.269342,47.2,1


In [4]:
import pandas as pd
import joblib

# ==========================
# Load Saved Model
# ==========================

best_model = joblib.load(r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\logistic_pipeline.pkl")
best_model2 = joblib.load(r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\random_forest_pipeline.pkl")
best_model3 = joblib.load(r"C:\Users\AiK\Machine Learning Projects\CCTV Maintanence\Model Building\Models\xgboost_pipeline.pkl")
# ==========================
# New CCTV Data
# ==========================

new_data = pd.DataFrame({

    "camera_id":[
        "CAM50001","CAM50002","CAM50003","CAM50004","CAM50005",
        "CAM50006","CAM50007","CAM50008","CAM50009","CAM50010"
    ],

    "age_days":[
        400,1100,120,800,1500,
        300,90,2000,650,170
    ],

    "daily_usage_hours":[
        16,19,14,18,20,
        17,15,21,18,16
    ],

    "avg_temperature_c":[
        29,41,22,37,43,
        25,18,40,34,26
    ],

    "avg_humidity_percent":[
        55,87,45,80,92,
        65,40,85,72,60
    ],

    "vibration_level":[
        0.5,3.2,0.2,1.6,4.5,
        0.4,0.1,4.2,1.1,0.3
    ],

    "signal_strength_dbm":[
        -55,-74,-49,-66,-79,
        -58,-46,-77,-60,-52
    ],

    "avg_power_consumption_w":[
        15,22,13,18,25,
        16,14,24,17,15
    ],

    "firmware_version":[
        4,2,7,5,1,
        8,10,3,6,9
    ],

    "camera_type":[
        "Dome","Bullet","PTZ","Dome","Bullet",
        "PTZ","Dome","Bullet","PTZ","Dome"
    ],

    "location_type":[
        "Indoor","Outdoor","Indoor","Outdoor","Outdoor",
        "Indoor","Indoor","Outdoor","Indoor","Outdoor"
    ],

    "manufacturer":[
        "Hikvision","Dahua","Axis","Bosch","Hikvision",
        "Axis","Bosch","Dahua","Hikvision","Axis"
    ],

    "weather_exposure":[
        "Low","High","Low","Medium","High",
        "Low","Low","High","Medium","Low"
    ]

})
# ==========================
# Feature Engineering
# ==========================

def create_features(df):

    df = df.copy()

    df["camera_age_years"] = df["age_days"] / 365

    df["old_camera"] = (df["age_days"] > 1000).astype(int)

    df["high_temp"] = (df["avg_temperature_c"] > 35).astype(int)

    df["high_humidity"] = (df["avg_humidity_percent"] > 80).astype(int)

    df["weak_signal"] = (df["signal_strength_dbm"] < -65).astype(int)

    df["high_usage"] = (df["daily_usage_hours"] > 18).astype(int)

    df["power_per_hour"] = (
        df["avg_power_consumption_w"] /
        df["daily_usage_hours"]
    )

    df["temp_humidity"] = (
        df["avg_temperature_c"] *
        df["avg_humidity_percent"]
    )

    df["age_vibration"] = (
        df["camera_age_years"] *
        df["vibration_level"]
    )

    df["signal_quality"] = (
        100 +
        df["signal_strength_dbm"]
    )

    df["risk_score"] = (
        df["old_camera"] +
        df["high_temp"] +
        df["high_humidity"] +
        df["weak_signal"] +
        df["high_usage"]
    )

    return df

new_data = create_features(new_data)

# ==========================
# Keep Camera IDs
# ==========================

camera_ids = new_data["camera_id"]

# ==========================
# Remove ID Column
# ==========================

prediction_data = new_data.drop(columns=["camera_id"])

# ==========================
# IMPORTANT
# Match Training Columns
# ==========================

expected_columns = best_model.feature_names_in_

prediction_data = prediction_data[expected_columns]

# ==========================
# Predict
# ==========================

predictions1 = best_model.predict(prediction_data)
probabilities1 = best_model.predict_proba(prediction_data)[:,1]
predictions2 = best_model2.predict(prediction_data)
probabilities2 = best_model2.predict_proba(prediction_data)[:,1]
predictions3 = best_model3.predict(prediction_data)
probabilities3 = best_model3.predict_proba(prediction_data)[:,1]
# ==========================
# Final Output
# ==========================

results = pd.DataFrame({
    
    "Camera ID":camera_ids,

    "Prediction":predictions1,

    "Status":[
        "Needs Maintenance" if x==1 else "Healthy"
        for x in predictions1
    ],

    "Maintenance Probability (%)":
        (probabilities1*100).round(2)

})
print("LOGISTIC REGRESSION")
print(results)

results = pd.DataFrame({
    
    "Camera ID":camera_ids,

    "Prediction":predictions2,

    "Status":[
        "Needs Maintenance" if x==1 else "Healthy"
        for x in predictions2
    ],

    "Maintenance Probability (%)":
        (probabilities2*100).round(2)

})
print("RANDOM FOREST")
print(results)


results = pd.DataFrame({
    
    "Camera ID":camera_ids,

    "Prediction":predictions3,

    "Status":[
        "Needs Maintenance" if x==1 else "Healthy"
        for x in predictions3
    ],

    "Maintenance Probability (%)":
        (probabilities3*100).round(2)

})
print("XGBOOST")
print(results)
# ==========================
# Save Results
# ==========================

results.to_csv(
    "predictions1.csv",
    index=False
)
results.to_csv(
    "predictions2.csv",
    index=False
)

results.to_csv(
    "predictions3.csv",
    index=False
)

print("\nPrediction file saved successfully!")

LOGISTIC REGRESSION
  Camera ID  Prediction             Status  Maintenance Probability (%)
0  CAM50001           0            Healthy                         8.27
1  CAM50002           0            Healthy                         0.00
2  CAM50003           0            Healthy                        49.99
3  CAM50004           0            Healthy                         0.01
4  CAM50005           0            Healthy                         0.00
5  CAM50006           0            Healthy                         3.51
6  CAM50007           1  Needs Maintenance                        64.21
7  CAM50008           0            Healthy                         0.00
8  CAM50009           0            Healthy                         0.11
9  CAM50010           0            Healthy                        34.45
RANDOM FOREST
  Camera ID  Prediction   Status  Maintenance Probability (%)
0  CAM50001           0  Healthy                         1.51
1  CAM50002           0  Healthy                  

In [7]:
print(best_model.named_steps)

{'prep': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imp',
                                                  SimpleImputer(strategy='median'))]),
                                 ['age_days', 'daily_usage_hours',
                                  'avg_temperature_c', 'avg_humidity_percent',
                                  'vibration_level', 'signal_strength_dbm',
                                  'avg_power_consumption_w', 'firmware_version',
                                  'camera_age_years', 'old_camera', 'high_temp',
                                  'high_humidity', 'weak_signal', 'high_usage',
                                  'power_per_hour', 'temp_humidity',
                                  'age_vibration', 'signal_quality',
                                  'risk_score']),
                                ('cat',
                                 Pipeline(steps=[('imp',
                                                  SimpleIm

In [6]:
import sklearn
print(sklearn.__version__)

1.0.2
